In [423]:
from box import Box

In [424]:
args = Box({
    'batch_size': 32,
    'cola_classifier_path': '/content/drive/MyDrive/style_transfer/cola_classifier',
    'wieting_tokenizer_path': 'sim.sp.30k.model',
    'wieting_model_path': 'sim.pt',
    't1': 75., # this is default value
    't2': 70., # this is default value
    't3': 12. # this is default value
})

In [425]:
epochs = 3
ckpt_num = 0
ckpt_num = 4642
lr = 2e-4
batch_size = 8
num_unmask_steps = 128
p_uncond = 0.0
cfg_scale = 1.0
num_cands = 4
instruct = True
svdd = False
baseline = False

instruct_str = '-instruct' if instruct else ''
svdd_str = '-svdd' if svdd else ''
num_cands_str = f'_{num_cands}numcands' if svdd else ''

if baseline:
    results_path = f'./results{instruct_str}/baseline.jsonl'
    output_path = f'./results{instruct_str}/baseline_eval.json'
else:
    if ckpt_num == 0:
        results_path = f'./results{instruct_str}{svdd_str}/llada_untrained/result_{num_unmask_steps}un_{cfg_scale}cfgscale{num_cands_str}.jsonl'
        output_path = f'./results{instruct_str}{svdd_str}/llada_untrained/eval_{num_unmask_steps}un_{cfg_scale}cfgscale{num_cands_str}.json'
    else:
        results_path = f'./results{instruct_str}{svdd_str}/llada_{epochs}ep-{batch_size}bs-{lr}lr-{p_uncond}puncond/result_{num_unmask_steps}un_{ckpt_num}ch_{cfg_scale}cfgscale{num_cands_str}.jsonl'
        output_path = f'./results{instruct_str}{svdd_str}/llada_{epochs}ep-{batch_size}bs-{lr}lr-{p_uncond}puncond/eval_{num_unmask_steps}un_{ckpt_num}ch_{cfg_scale}cfgscale{num_cands_str}.json'
print("Results will be saved to:", output_path)

Results will be saved to: ./results-instruct/llada_3ep-8bs-0.0002lr-0.0puncond/eval_128un_4642ch_1.0cfgscale.json


In [426]:
import json

preds = []
actuals = []
hates = []

with open(results_path, 'r') as f:
	for line in f:
		result = json.loads(line)
		# if "Read the following article and answer the question." not in result['pred']:
		# 	continue
		pred = result['pred']
		pred = pred.split('<|endoftext|>')[0].strip()
		preds.append(pred.lower())
		actual = result['actual']
		actual = actual.split('<|endoftext|>')[0].strip()
		actuals.append(actual.lower())
		hate = result['hate']
		hates.append(hate.lower())

In [427]:
len(preds), len(actuals), len(hates)

(198, 198, 198)

In [428]:
import torch

### Style Transfer Accuracy metric

In [429]:
from paradetox.evaluation_detox.metric_tools.style_transfer_accuracy import classify_preds
import numpy as np

In [430]:
accuracy_by_sent = classify_preds(args, preds)
accuracy = np.mean(accuracy_by_sent)
print('\n')
accuracy_by_sent = torch.tensor(accuracy_by_sent)
print(accuracy)
torch.where(accuracy_by_sent == 0)

Calculating style of predictions


Some weights of the model checkpoint at SkolkovoInstitute/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 7/7 [00:02<00:00,  2.99it/s]



0.9696969696969697


(tensor([ 30,  58,  91, 151, 152, 189]),)

### Content Similarity metrics

In [431]:
from paradetox.evaluation_detox.metric_tools.content_similarity import flair_sim

In [432]:
emb_sim_stats = flair_sim(args, hates, preds)
emb_sim = emb_sim_stats.mean()
print('\n')
print(emb_sim_stats)
print(emb_sim)

Calculating flair embeddings similarity


tensor([0.8770, 0.9427, 0.8018, 0.8266, 0.8686, 0.9869, 0.7503, 0.9044, 0.5998,
        0.9252, 0.8992, 0.8403, 0.8139, 0.8889, 0.6835, 0.8241, 0.8470, 0.9212,
        0.8283, 0.8795, 0.4753, 0.9116, 0.8142, 0.8549, 0.7927, 0.9324, 0.8797,
        0.7695, 0.8627, 0.9515, 0.8485, 0.9159, 0.8215, 0.7806, 0.9155, 0.8971,
        0.9767, 0.7792, 0.7634, 0.9847, 0.9443, 0.8844, 0.7514, 0.9572, 0.7834,
        0.7272, 0.8052, 0.9138, 0.8178, 0.8562, 0.7426, 0.9582, 0.9279, 0.7919,
        0.6911, 0.9387, 0.7659, 0.9156, 0.7842, 0.8604, 0.8783, 0.9845, 0.5798,
        0.8038, 0.8616, 0.8974, 0.8547, 0.6341, 0.7884, 0.8933, 0.7862, 0.9837,
        0.8878, 0.9786, 0.7879, 0.7007, 0.9938, 0.9175, 0.9769, 0.8225, 0.8837,
        0.8745, 0.9698, 0.9577, 0.9344, 0.7137, 0.8264, 0.6805, 0.6242, 0.8892,
        0.9237, 0.9504, 0.9004, 0.9128, 0.8718, 0.8586, 0.9194, 0.9012, 0.9312,
        0.9562, 0.5985, 0.8160, 0.7525, 0.9257, 0.8888, 0.9156, 0.9383, 0.9234

### Fluency metrics

In [433]:
from paradetox.evaluation_detox.metric_tools.fluency import cola_fluency

In [434]:
cola_stats = cola_fluency(preds)
cola_acc = sum(cola_stats) / len(preds)
print('\n')
cola_stats = torch.tensor(cola_stats)
print(cola_stats)
print(cola_acc)



tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        0, 1, 1, 1, 1, 1])
0.9292929292929293


### Fluency similarity metrics

In [435]:
cola_stats_hates = cola_fluency(hates)
# cola_acc_hates = sum(cola_stats) / len(preds)
print('\n')
cola_stats_hates = torch.tensor(cola_stats_hates)
cola_sim_accuracy = torch.sum(cola_stats == cola_stats_hates)/cola_stats.shape[0]
print(cola_sim_accuracy)



tensor(0.7424)


### Joint metrics

In [436]:
from paradetox.evaluation_detox.metric_tools.joint_metrics import *

In [437]:
joint = get_j(args, accuracy_by_sent, emb_sim_stats, cola_stats, preds)
print(joint)

tensor(0.7608)


In [438]:
joint_with_fl_acc = get_j(args, accuracy_by_sent, emb_sim_stats, cola_sim_accuracy, preds)

### Saving Results

In [439]:
eval_dict = {
    "STA": float(accuracy),
	"SIM": float(emb_sim),
    "FL": float(cola_acc),
    "FL_acc": float(cola_sim_accuracy),
	"J": float(joint),
    "J_with_FL_acc": float(joint_with_fl_acc)
}

In [440]:
with open(output_path, 'w') as f:
	json.dump(eval_dict, f, indent=4)